### PSD Pipeline
#### Table of Contents:
* [1. Environment Setup](#1-environment-setup)
    * [1.1 Library Imports](#11-library-imports)
    * [1.2 Dataset loading/inspection](#12-dataset-loadinginspection)
* [2. Data Segmentation](#data-segmentation)
* [3. Feature Extraction and Class Distribution](#3-feature-extraction--class-distribution)
    * [3.1 DWT](#31-dwt-feature-extraction)
    * [3.2 Class Distribution](#32-class-distribution)
* [4. Classification Schemes](#4-classification-schemes)
    * [4.1 Emotional vs. Neutral](#41-emotional-vs-neutral-remap--class-distribution)
    * [4.2 Positive vs. Negative](#42-positive-vs-negative-remap--class-distribution)
* [5. Model Training](#5-model-training)
    * [5.1 XGBoost](#51-xgboost)
        * [Emotional vs. Neutral](#511-emotional-vs-neutral)




### 1. Environment Setup

##### 1.1 Library Imports

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.io import loadmat
import pywt as pwt
from pywt import wavedec

import matplotlib.pyplot as plt
import seaborn as sns

import optuna
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import LeaveOneGroupOut, StratifiedKFold, StratifiedGroupKFold
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, roc_auc_score, balanced_accuracy_score, classification_report


##### 1.2 Dataset loading/inspection

In [ ]:
dataset_path = Path('DEED')
eeg_dataset = []
subject_ids = []

for file in dataset_path.iterdir():
    mat = loadmat(file)
    eeg = mat['Data']
    fname = file.stem  
    label_part = [part for part in fname.split("_") if part.startswith("E")][0]
    subject_part = [part for part in fname.split("_") if part.startswith("S")][0]
    label = int(label_part[1:])  
    subject_id = subject_part[1:-1]  # last two digits = subject number
    
    eeg_dataset.append((eeg, label))
    subject_ids.append(subject_id)

print(f"Loaded {len(eeg_dataset)} trials.")
print(f"Unique subjects: {len(set(subject_ids))}")
print(f"Subject IDs: {sorted(set(subject_ids))}")
print("Example shapes:", [(arr.shape, lbl) for arr, lbl in eeg_dataset[:3]])

print("\nSample filename → label mapping:")
for file, (_, label) in zip(dataset_path.iterdir(), eeg_dataset[:10]):
    print(f"  {file.stem} → E{label}")

### 2. Data Segmentation

Done into 20s windows.

In [ ]:
def segmentation(eeg_dataset, subject_ids, window_sec, fs):
    window_size = int(window_sec * fs)
    X = []
    y = []
    groups = []

    for (eeg_array, label), sid in zip(eeg_dataset, subject_ids):
        n_samples = eeg_array.shape[1]
        start = 0
        while start + window_size <= n_samples:
            window = eeg_array[:, start:start + window_size]
            X.append(window)
            y.append(label)
            groups.append(sid)
            start += window_size  

    return X, y, groups

windows, window_labels, window_groups = segmentation(eeg_dataset, subject_ids, 20, 200)
print(f"Total 20 second windows: {len(windows)}")
print(f"Unique subjects: {len(set(window_groups))}")

### 3. Feature Extraction & Class Distribution

DWT features are extracted from each 20s window decomposed to level 5. For each channel, energy, variance, standard deviation, and peak amplitude are computed across all coefficient arrays, alongside hemispheric asymmetry features for each frontal, frontotemporal, and temporal channel pair, yielding 198 features per window.

##### 3.1 DWT Feature Extraction

In [ ]:
CHANNELS = ['F3', 'F4', 'FT7', 'FT8', 'T7', 'T8']
ASYM_PAIRS = [(0, 1), (2, 3), (4, 5)]  # F3/F4, FT7/FT8, T7/T8

def extract_dwt_features(segmented_windows, labels):
    features = []
    for window in segmented_windows:
        channel_features = []
        channel_coeffs = []

        # Per-channel per-band stats
        for ch in range(window.shape[0]):
            coeffs = wavedec(window[ch], 'db4', level=5)
            channel_coeffs.append(coeffs)
            for coeff in coeffs:
                channel_features.extend([
                    np.mean(coeff),
                    np.std(coeff),
                    np.var(coeff),
                    np.sum(coeff**2),        # band energy
                    np.max(np.abs(coeff)),   # peak amplitude
                ])

        for left, right in ASYM_PAIRS:
            for level in range(6):  # 6 coefficient arrays at level=5
                left_energy = np.sum(channel_coeffs[left][level]**2)
                right_energy = np.sum(channel_coeffs[right][level]**2)
                asymmetry = (left_energy - right_energy) / (left_energy + right_energy + 1e-8)
                channel_features.append(asymmetry)

        features.append(channel_features)

    X = np.array(features)
    y = np.array(labels)
    return X, y


##### 3.2 Class Distribution

In [ ]:
X, y = extract_dwt_features(windows, window_labels)
print(X.shape)
print(y.shape) 

print("\n=== Total Class Distribution===")
unique, counts = np.unique(y, return_counts=True)
for label, count in zip(unique, counts):
    print(f"E{label}: {count} windows ({count/len(y)*100:.1f}%)")

In [ ]:
unique, counts = np.unique(y, return_counts=True)
total = len(y)

classes = ['E0\n(Dreamless)', 'E1\n(Negative)', 'E2\n(Rel. Negative)',
           'E3\n(Neutral)', 'E4\n(Rel. Positive)', 'E5\n(Positive)']

sns.set_theme(style="ticks", font_scale=1.1)
fig, ax = plt.subplots(figsize=(9, 5))

bars = ax.bar(classes, counts, color='#5b8db8', edgecolor='white', linewidth=0.8, width=0.6)

for bar, count in zip(bars, counts):
    pct = count / total * 100
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 40,
            f'{count}\n({pct:.1f}%)',
            ha='center', va='bottom', fontsize=9.5, color='#333333')

ax.set_ylabel('Number of Windows', fontsize=11, labelpad=10)
ax.set_title('Class Distribution Across All Emotion Labels', fontsize=13, fontweight='bold', pad=15)
ax.set_ylim(0, max(counts) * 1.22)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.tick_params(axis='x', labelsize=9.5)
ax.tick_params(axis='y', labelsize=9.5)

plt.tight_layout()
plt.savefig('class_dist_full.png', dpi=300, bbox_inches='tight')
plt.show()

### 4. Classification Schemes

##### 4.1 Emotional vs. Neutral (remap + class distribution)  
Emotional {E1, E2, E4, E5} vs. Neutral Dream {E3} 

In [ ]:
def remap_emotional_neutral(y):
    y = np.array(y)
    keep_mask = np.isin(y, [1, 2, 3, 4, 5])  
    new_y = np.zeros(len(y), dtype=int)
    new_y[np.isin(y, [1, 2, 4, 5])] = 1  
    return new_y[keep_mask], keep_mask

y_en, mask_en = remap_emotional_neutral(y)
X_en = X[mask_en]
groups_en = np.array(window_groups)[mask_en]

print(f"\n=== Emotional vs. Neutral Class Distribution ===")
print(f"  Total: {len(y_en)}")
print(f"  Neutral (0): {np.sum(y_en == 0)} ({np.sum(y_en == 0)/len(y_en)*100:.1f}%)")
print(f"  Emotional (1): {np.sum(y_en == 1)} ({np.sum(y_en == 1)/len(y_en)*100:.1f}%)")

In [ ]:
# Subject Wise Class Distribution (EN)
print("=== Subject Wise Class Distribution (EN) ===")
for subject in sorted(set(groups_en)):
    mask = groups_en == subject
    labels = y_en[mask]
    unique = np.unique(labels)
    print(f"Subject {subject}: {len(labels)} windows, classes: {unique}, counts: {np.bincount(labels)}")

##### 4.2 Positive vs. Negative (remap + class distribution)
Positive {E4, E5} vs. Negative {E1, E2}

In [ ]:
def remap_positive_negative(y):
    y = np.array(y)
    keep_mask = np.isin(y, [1, 2, 4, 5])  
    new_y = np.zeros(len(y), dtype=int)
    new_y[np.isin(y, [4, 5])] = 1  
    return new_y[keep_mask], keep_mask

y_pn, mask_pn = remap_positive_negative(y)
X_pn = X[mask_pn]
groups_pn = np.array(window_groups)[mask_pn]

print(f"\n=== Positive vs. Negative Class Distribution ===")
print(f"  Total: {len(y_pn)}")
print(f"  Negative (0): {np.sum(y_pn == 0)} ({np.sum(y_pn == 0)/len(y_pn)*100:.1f}%)")
print(f"  Positive (1): {np.sum(y_pn == 1)} ({np.sum(y_pn == 1)/len(y_pn)*100:.1f}%)")

In [ ]:
# Subject Wise Class Distribution (PN)
print("=== Subject Wise Class Distribution (PN) ===")
for subject in sorted(set(groups_pn)):
    mask = groups_pn == subject
    labels = y_pn[mask]
    unique = np.unique(labels)
    print(f"Subject {subject}: {len(labels)} windows, classes: {unique}, counts: {np.bincount(labels)}")

In [ ]:
sns.set_theme(style="ticks", font_scale=1.1)
fig, (ax2, ax3) = plt.subplots(1, 2, figsize=(9, 5))

# Positive vs Negative
pn_counts = [np.sum(y_pn == 0), np.sum(y_pn == 1)]
pn_labels = ['Negative (0)', 'Positive (1)']
bars2 = ax2.bar(pn_labels, pn_counts, color='#5b8db8', edgecolor='white', linewidth=0.8, width=0.5)
for bar, count in zip(bars2, pn_counts):
    pct = count / sum(pn_counts) * 100
    ax2.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 20,
             f'{count}\n({pct:.1f}%)',
             ha='center', va='bottom', fontsize=9.5, color='#333333')
ax2.set_title('Positive vs. Negative', fontsize=11, fontweight='bold', pad=12)
ax2.set_ylabel('Number of Windows', fontsize=10, labelpad=10)
ax2.set_ylim(0, max(pn_counts) * 1.25)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.spines['left'].set_visible(False)
ax2.tick_params(axis='x', labelsize=9.5)
ax2.tick_params(axis='y', labelsize=9.5)

# Emotional vs Neutral
en_counts = [np.sum(y_en == 0), np.sum(y_en == 1)]
en_labels = ['Neutral (0)', 'Emotional (1)']
bars3 = ax3.bar(en_labels, en_counts, color='#5b8db8', edgecolor='white', linewidth=0.8, width=0.5)
for bar, count in zip(bars3, en_counts):
    pct = count / sum(en_counts) * 100
    ax3.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 20,
             f'{count}\n({pct:.1f}%)',
             ha='center', va='bottom', fontsize=9.5, color='#333333')
ax3.set_title('Emotional vs. Neutral', fontsize=11, fontweight='bold', pad=12)
ax3.set_ylabel('Number of Windows', fontsize=10, labelpad=10)
ax3.set_ylim(0, max(en_counts) * 1.25)
ax3.spines['top'].set_visible(False)
ax3.spines['right'].set_visible(False)
ax3.spines['left'].set_visible(False)
ax3.tick_params(axis='x', labelsize=9.5)
ax3.tick_params(axis='y', labelsize=9.5)

fig.suptitle('Binary Classification Task Distributions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('class_dist_binary.png', dpi=300, bbox_inches='tight')
plt.show()

### 5. Model Training
For both models and classification schemes, hyperparameters are optimized using Optuna with 5-fold StratifiedGroupKFold on the full dataset. The resulting best parameters are fixed and used for final evaluation with LOSO. For comparison, performance is also assessed using standard 10-fold cross-validation.

#### 5.1 XGBoost

##### General Functions for Training

In [ ]:
optuna.logging.set_verbosity(optuna.logging.INFO)

def xgb_hyperparameter_training_loso(X, y, groups, model_name, classification_scheme):
    print(f"=== Hyperparameter Tuning - {model_name} ({classification_scheme}) ===")

    def objective(trial):
        params = {
            'n_estimators':     trial.suggest_int('n_estimators', 100, 600),
            'max_depth':        trial.suggest_int('max_depth', 3, 8),
            'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
            'gamma':            trial.suggest_float('gamma', 0, 5),
            'eval_metric':      'logloss',
        }
        cv = StratifiedGroupKFold(n_splits=5)
        scores = []
        for train_idx, val_idx in cv.split(X, y, groups):
            X_train, X_val = X[train_idx], X[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]

            neg = np.sum(y_train == 0)
            pos = np.sum(y_train == 1)
            params['scale_pos_weight'] = neg/pos

            model = XGBClassifier(**params, n_jobs=-1, random_state = 42)
            model.fit(X_train, y_train)
            preds = model.predict(X_val)

            scores.append(f1_score(y_val, preds, average='macro'))

        return np.mean(scores)

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials = 50)
    print(f"Best params: {study.best_params}")
    print(f"Best CV F1: {study.best_value:.4f}")


    # Save best params
    best_params = study.best_params
    best_params['random_state'] = 42
    best_params['n_jobs'] = -1
    return best_params

In [ ]:
optuna.logging.set_verbosity(optuna.logging.INFO)

def xgb_hyperparameter_training_10f(X, y,model_name, classification_scheme):
    print(f"=== Hyperparameter Tuning - {model_name} ({classification_scheme}) ===")

    def objective(trial):
        params = {
            'n_estimators':     trial.suggest_int('n_estimators', 100, 600),
            'max_depth':        trial.suggest_int('max_depth', 3, 8),
            'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
            'gamma':            trial.suggest_float('gamma', 0, 5),
            'eval_metric':      'logloss',
        }
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        scores = []
        for train_idx, val_idx in cv.split(X, y):
            X_train, X_val = X[train_idx], X[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]

            neg = np.sum(y_train == 0)
            pos = np.sum(y_train == 1)
            params['scale_pos_weight'] = neg/pos

            model = XGBClassifier(**params, n_jobs=-1, random_state = 42)
            model.fit(X_train, y_train)
            preds = model.predict(X_val)

            scores.append(f1_score(y_val, preds, average='macro'))

        return np.mean(scores)

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials = 15)
    print(f"Best params: {study.best_params}")
    print(f"Best CV F1: {study.best_value:.4f}")


    # Save best params
    best_params = study.best_params
    best_params['random_state'] = 42
    best_params['n_jobs'] = -1
    return best_params

In [ ]:
def xgb_loso_loop(X, y, groups, params, model_name, classification_scheme):
    # LOSO evaluation with fixed params
    print(f"\n=== LOSO - {model_name} ({classification_scheme}) ===")
    logo = LeaveOneGroupOut()
    accs = []
    f1s = []
    aurocs = []
    bal_accs = []

    for fold, (train_idx, test_idx) in enumerate(logo.split(X, y, groups)):
        if len(np.unique(y[test_idx])) < 2:
            continue

        subject = groups[test_idx[0]]
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        #Per fold class weighting
        neg = np.sum(y_train == 0)
        pos = np.sum(y_train == 1)

        fold_params = params.copy()
        fold_params['scale_pos_weight'] = neg/pos

        model = XGBClassifier(**fold_params)
        model.fit(X_train, y_train)

        preds = model.predict(X_test)
        proba = model.predict_proba(X_test)[:, 1]

        accs.append(accuracy_score(y_test, preds))
        f1s.append(f1_score(y_test, preds, average='macro'))
        aurocs.append(roc_auc_score(y_test, proba))
        bal_accs.append(balanced_accuracy_score(y_test, preds))
        print(f"Subject {subject} | Balanced Accuracy: {bal_accs[-1]:.4f} | Accuracy: {accs[-1]:.4f} | F1: {f1s[-1]:.4f} | AUROC: {aurocs[-1]:.4f}")

    print(f"Balanced Accuracy:  {np.mean(bal_accs):.4f} ± {np.std(bal_accs):.4f}")
    print(f"Accuracy:  {np.mean(accs):.4f} ± {np.std(accs):.4f}")
    print(f"F1:  {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
    print(f"AUROC:  {np.mean(aurocs):.4f} ± {np.std(aurocs):.4f}")


In [ ]:
def xgb_ten_fold_cv_loop(X, y, params, model_name, classification_scheme):
    print(f"\n=== 10-Fold CV - {model_name} ({classification_scheme}) ===")
    # 10 Fold Cross CV with same tuned params
    cv_10fold = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    accs = []
    f1s = []
    bal_accs = []
    aurocs = []

    for train_idx, test_idx in cv_10fold.split(X, y):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        neg = np.sum(y_train == 0)
        pos = np.sum(y_train == 1)
        fold_params = params.copy()
        fold_params['scale_pos_weight'] = neg / pos

        model = XGBClassifier(**fold_params)
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        proba = model.predict_proba(X_test)[:, 1]

        accs.append(accuracy_score(y_test, preds))
        f1s.append(f1_score(y_test, preds, average='macro'))
        aurocs.append(roc_auc_score(y_test, proba))
        bal_accs.append(balanced_accuracy_score(y_test, preds))

    print(f"Accuracy:          {np.mean(accs):.4f} ± {np.std(accs):.4f}")
    print(f"F1:                {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
    print(f"Balanced Accuracy: {np.mean(bal_accs):.4f} ± {np.std(bal_accs):.4f}")
    print(f"AUROC:             {np.mean(aurocs):.4f} ± {np.std(aurocs):.4f}")

##### 5.1.1 Emotional vs. Neutral

In [ ]:
#Hyperparameter tuning for EN LOSO
xgb_en_params_loso = xgb_hyperparameter_training_loso(X_en, y_en, groups_en, "XGBoost", "Emotional vs. Neutral")

In [ ]:
xgb_en_params_10f = xgb_hyperparameter_training_loso(X_en, y_en, "XGBoost", "Emotional vs. Neutral")

In [ ]:
#LOSO Evaluation
xgb_en_loso = xgb_loso_loop(X_en, y_en, groups_en, xgb_en_params_loso, "XGBoost", "Emotional vs. Neutral")

In [ ]:
#10Fold CV
xgb_en_10f = xgb_ten_fold_cv_loop(X_en, y_en, xgb_en_params_10f, "XGBoost", "Emotional vs. Neutral")

##### 5.1.2 Positive vs. Negative

In [ ]:
#Hyperparameter tuning
xgb_pn_params_loso = xgb_hyperparameter_training_loso(X_pn, y_pn, groups_pn, "XGBoost", "Positive vs. Negative")

In [ ]:
xgb_pn_params_10f = xgb_hyperparameter_training_loso(X_pn, y_pn, "XGBoost", "Positive vs. Negative")

In [ ]:
#LOSO Evaluation
xgb_pn_loso = xgb_loso_loop(X_pn, y_pn, groups_pn, xgb_pn_params_loso, "XGBoost", "Positive vs. Negative")

In [ ]:
#10Fold CV
xgb_pn_10f = xgb_ten_fold_cv_loop(X_pn, y_pn, xgb_pn_params_10f, "XGBoost", "Positive vs. Negative")

#### 5.2 KNN (K-Nearest Neighbors)

##### General Functions for Training

In [ ]:
def subject_normalize(X, groups):
    X_norm = X.copy()
    for subj in np.unique(groups):
        mask = groups == subj
        scaler = StandardScaler()
        X_norm[mask] = scaler.fit_transform(X[mask])
    return X_norm

In [ ]:
optuna.logging.set_verbosity(optuna.logging.INFO)

def knn_hyperparameter_training_loso(X, y, groups, model_name, classification_scheme):
    print(f"=== Hyperparameter Tuning - {model_name} ({classification_scheme}) ===")

    def objective(trial):
        params = {
            'n_neighbors': trial.suggest_int('n_neighbors', 1, 30),
            'weights': trial.suggest_categorical('weights', ['uniform', 'distance']),
            'metric': trial.suggest_categorical('metric', ['euclidean', 'manhattan', 'cosine']),
            'leaf_size': trial.suggest_int('leaf_size', 10, 50),
        }
        cv = StratifiedGroupKFold(n_splits=5)
        scores = []
        for train_idx, val_idx in cv.split(X, y, groups):
            X_train, X_val = X[train_idx], X[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]
            
            groups_train = groups[train_idx]
            groups_val   = groups[val_idx]

            X_train_norm = subject_normalize(X_train, groups_train)
            X_val_norm   = subject_normalize(X_val, groups_val)

            model = KNeighborsClassifier(**params)
            model.fit(X_train_norm, y_train)
            preds = model.predict(X_val_norm)
            scores.append(f1_score(y_val, preds, average='macro'))

        return np.mean(scores)

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials = 50)
    print(f"Best params: {study.best_params}")
    print(f"Best CV F1: {study.best_value:.4f}")


    # Save best params
    best_params = study.best_params
    return best_params

In [ ]:
def knn_hyperparameter_training_10f(X, y, model_name, classification_scheme):
    print(f"=== Hyperparameter Tuning (10-Fold) - {model_name} ({classification_scheme}) ===")

    def objective(trial):
        params = {
            'n_neighbors': trial.suggest_int('n_neighbors', 1, 30),
            'weights': trial.suggest_categorical('weights', ['uniform', 'distance']),
            'metric': trial.suggest_categorical('metric', ['euclidean', 'manhattan', 'cosine']),
            'leaf_size': trial.suggest_int('leaf_size', 10, 50),
        }
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        scores = []
        for train_idx, val_idx in cv.split(X, y):
            X_train, X_val = X[train_idx], X[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]

            scaler = StandardScaler()
            X_train = scaler.fit_transform(X_train)
            X_val = scaler.transform(X_val)

            model = KNeighborsClassifier(**params)
            model.fit(X_train, y_train)
            preds = model.predict(X_val)
            scores.append(f1_score(y_val, preds, average='macro'))

        return np.mean(scores)

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=50)
    print(f"Best params: {study.best_params}")
    print(f"Best CV F1: {study.best_value:.4f}")

    return study.best_params

In [ ]:
def knn_loso_loop(X, y, groups, params, model_name, classification_scheme):
    # LOSO evaluation with fixed params
    print(f"\n=== LOSO - {model_name} ({classification_scheme}) ===")
    logo = LeaveOneGroupOut()
    accs = []
    f1s = []
    aurocs = []
    bal_accs = []

    for fold, (train_idx, test_idx) in enumerate(logo.split(X, y, groups)):
        if len(np.unique(y[test_idx])) < 2:
            continue

        subject = groups[test_idx[0]]
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        groups_train = groups[train_idx]
        groups_test  = groups[test_idx]

        X_train_norm = subject_normalize(X_train, groups_train)
        X_test_norm  = subject_normalize(X_test, groups_test)

        model = KNeighborsClassifier(**params)
        model.fit(X_train_norm, y_train)
        preds = model.predict(X_test_norm)
        proba = model.predict_proba(X_test_norm)[:, 1]

        accs.append(accuracy_score(y_test, preds))
        f1s.append(f1_score(y_test, preds, average='macro'))
        aurocs.append(roc_auc_score(y_test, proba))
        bal_accs.append(balanced_accuracy_score(y_test, preds))
        print(f"Subject {subject} | Balanced Accuracy: {bal_accs[-1]:.4f} | Accuracy: {accs[-1]:.4f} | F1: {f1s[-1]:.4f} | AUROC: {aurocs[-1]:.4f}")

    print(f"Balanced Accuracy:  {np.mean(bal_accs):.4f} ± {np.std(bal_accs):.4f}")
    print(f"Accuracy:  {np.mean(accs):.4f} ± {np.std(accs):.4f}")
    print(f"F1:  {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
    print(f"AUROC:  {np.mean(aurocs):.4f} ± {np.std(aurocs):.4f}")


In [ ]:
def knn_ten_fold_cv_loop(X, y, params, model_name, classification_scheme):
    print(f"\n=== 10-Fold CV - {model_name} ({classification_scheme}) ===")
    # 10 Fold Cross CV with same tuned params
    cv_10fold = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    accs = []
    f1s = []
    bal_accs = []
    aurocs = []

    for train_idx, test_idx in cv_10fold.split(X, y):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        model = KNeighborsClassifier(**params)
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        proba = model.predict_proba(X_test)[:, 1]

        accs.append(accuracy_score(y_test, preds))
        f1s.append(f1_score(y_test, preds, average='macro'))
        aurocs.append(roc_auc_score(y_test, proba))
        bal_accs.append(balanced_accuracy_score(y_test, preds))

    print(f"Accuracy:          {np.mean(accs):.4f} ± {np.std(accs):.4f}")
    print(f"F1:                {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
    print(f"Balanced Accuracy: {np.mean(bal_accs):.4f} ± {np.std(bal_accs):.4f}")
    print(f"AUROC:             {np.mean(aurocs):.4f} ± {np.std(aurocs):.4f}")

##### 5.2.1 Emotional vs. Neutral

In [ ]:
#Hyperparameter tuning
knn_en_params_loso = knn_hyperparameter_training_loso(X_en, y_en, groups_en, "KNN", "Emotional vs. Neutral")

In [ ]:
knn_en_params_10f = knn_hyperparameter_training_10f(X_en, y_en, "KNN", "Emotional vs. Neutral")

In [ ]:
#LOSO Evaluation
knn_en_loso = knn_loso_loop(X_en, y_en, groups_en, knn_en_params_loso, "KNN", "Emotional vs. Neutral")

In [ ]:
#10Fold CV
knn_en_10f = knn_ten_fold_cv_loop(X_en, y_en, knn_en_params_10f, "KNN", "Emotional vs. Neutral")

##### 5.2.2 Positive vs. Negative

In [ ]:
#Hyperparameter tuning
knn_pn_params_loso = knn_hyperparameter_training_loso(X_pn, y_pn, groups_pn, "KNN", "Positive vs. Negative")

In [ ]:
knn_pn_params_10f = knn_hyperparameter_training_10f(X_pn, y_pn, "KNN", "Positive vs. Negative")

In [ ]:
#LOSO Evaluation
knn_pn_loso = knn_loso_loop(X_pn, y_pn, groups_pn, knn_pn_params_loso, "KNN", "Positive vs. Negative")

In [ ]:
#10Fold CV
knn_pn_10f = knn_ten_fold_cv_loop(X_pn, y_pn, knn_pn_params_10f, "KNN", "Positive vs. Negative")